# 03 — Apartes relacionais, segmentação e atos de fala

Analisa díades, constrói pontes para o corpus e prepara a análise qualitativa dos apartes e respostas.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/falando_nela/data")
REPO_DIR = Path("/content/falando_nela")
REPO_URL = "https://github.com/pedblan/falando_nela.git"
REPO_REF = ""  # Opcional: branch, tag ou commit; vazio acompanha o default remoto.

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags", "--prune"], check=True)
    if not REPO_REF:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
if REPO_REF:
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)

os.chdir(REPO_DIR)
os.environ["FALANDO_NELA_DATA_ROOT"] = str(DATA_ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-analise.txt"], check=True)
print("Data root:", DATA_ROOT)
print("Commit:", subprocess.run(["git", "rev-parse", "HEAD"], check=True, text=True, capture_output=True).stdout.strip())

## Configuração

Use o mesmo `RUN_ID` em toda a suíte. A configuração versionada é a fonte de verdade.

In [ ]:
from analise.discursos_plenario.config import load_config, resolve_input_paths, resolve_output_root

RUN_ID = "analise-plenario-20260713-v1"
CONFIG_PATH = REPO_DIR / "analise" / "discursos_plenario" / "config.v1.json"
ANALYSIS_CONFIG = load_config(CONFIG_PATH)
RUN_OUTPUT_ROOT = resolve_output_root(ANALYSIS_CONFIG, DATA_ROOT, RUN_ID)
INPUT_PATHS = resolve_input_paths(ANALYSIS_CONFIG, DATA_ROOT)
RODAR_ETAPA = False

assert ANALYSIS_CONFIG.date_start == "2010-02-02"
assert ANALYSIS_CONFIG.date_end == "2026-07-13"
assert ANALYSIS_CONFIG.raw["complete_year_end"] == 2025
assert ANALYSIS_CONFIG.raw["ytd_year"] == 2026
print("Run:", RUN_ID)
print("Saida:", RUN_OUTPUT_ROOT)

## Decisão metodológica

Sem precisão contra conjunto ouro, a ponte não autoriza denominadores e a segmentação não autoriza classificação. A taxonomia de atos de fala reproduz o TD 355 e permanece revisável.

In [ ]:
APARTES_PATH = INPUT_PATHS["interjections"]
APARTES_SNAPSHOT_PATH = RUN_OUTPUT_ROOT / "00_snapshot" / "discursos_plenario_snapshot.parquet"
assert APARTES_PATH.exists(), APARTES_PATH
assert APARTES_SNAPSHOT_PATH.exists(), "Execute o caderno 00."
VALIDAR_SEGMENTACAO = False
GERAR_JSONL_ATOS_FALA = False
ENVIAR_BATCH_ATOS_FALA = False
BAIXAR_BATCH_ATOS_FALA = False
PROCESSAR_BATCH_ATOS_FALA = False
APARTES_QUALITATIVE_MODEL = ANALYSIS_CONFIG.raw["openai"]["interjection_default_model"]

## Execução

A etapa cara permanece desativada até a inspeção das entradas e dos parâmetros acima.

In [ ]:
from analise.discursos_plenario.apartes import run_interjection_analysis

APARTES_RESULT = None
if RODAR_ETAPA:
    APARTES_RESULT = run_interjection_analysis(data_root=DATA_ROOT, run_id=RUN_ID, config_path=CONFIG_PATH)
    print(APARTES_RESULT["manifest_path"])
else:
    print("Apartes não executados.")

## Validação imediata

Esta checagem não substitui os testes sintéticos nem a revisão dos manifests.

In [ ]:
import json
import pandas as pd

APARTES_TESTS_PATH = RUN_OUTPUT_ROOT / "03_apartes" / "testes_associacao.csv"
APARTES_BRIDGE_QUALITY_PATH = RUN_OUTPUT_ROOT / "03_apartes" / "ponte_camara_qualidade.json"
APARTES_SEGMENTATION_QUALITY_PATH = RUN_OUTPUT_ROOT / "03_apartes" / "segmentacao_qualidade.json"
if APARTES_TESTS_PATH.exists():
    APARTES_TESTS = pd.read_csv(APARTES_TESTS_PATH)
    display(APARTES_TESTS.tail())
if APARTES_BRIDGE_QUALITY_PATH.exists():
    APARTES_BRIDGE_QUALITY = json.loads(APARTES_BRIDGE_QUALITY_PATH.read_text(encoding="utf-8"))
    assert APARTES_BRIDGE_QUALITY["denominators_authorized"] is False
    print(APARTES_BRIDGE_QUALITY)
if APARTES_SEGMENTATION_QUALITY_PATH.exists():
    APARTES_SEGMENTATION_QUALITY = json.loads(APARTES_SEGMENTATION_QUALITY_PATH.read_text(encoding="utf-8"))
    print(APARTES_SEGMENTATION_QUALITY)

## Revisar a segmentação dos turnos

O caderno cria uma amostra balanceada de 200 interações. Revise se o trecho é realmente o aparte e se a resposta pertence ao orador principal; só então recalcule a qualidade.

In [ ]:
import json
import pandas as pd
from analise.discursos_plenario.apartes_qualitativos import segmentation_quality
from analise.discursos_plenario.io import write_json_atomic

APARTES_INTERACTIONS_PATH = RUN_OUTPUT_ROOT / "03_apartes" / "interacoes_segmentadas.parquet"
APARTES_SEGMENTATION_REVIEW_PATH = RUN_OUTPUT_ROOT / "03_apartes" / "revisao_segmentacao.csv"
APARTES_SEGMENTATION_VALIDATION = None
if VALIDAR_SEGMENTACAO:
    APARTES_INTERACTIONS = pd.read_parquet(APARTES_INTERACTIONS_PATH)
    APARTES_SEGMENTATION_GOLD = pd.read_csv(APARTES_SEGMENTATION_REVIEW_PATH)
    APARTES_SEGMENTATION_VALIDATION = segmentation_quality(
        APARTES_INTERACTIONS,
        APARTES_SEGMENTATION_GOLD,
        min_precision=0.95,
        min_reviewed=100,
    )
    write_json_atomic(APARTES_SEGMENTATION_QUALITY_PATH, APARTES_SEGMENTATION_VALIDATION)
    print(APARTES_SEGMENTATION_VALIDATION)
else:
    print("Validação desativada; preencha primeiro a amostra de revisão.")

## Preparar atos de fala e possível descortesia

Complete o codebook do TD 355. O JSONL contém somente os turnos segmentados; respostas ausentes ficam explicitamente marcadas.

In [ ]:
import pandas as pd
from analise.discursos_plenario.apartes_qualitativos import write_qualitative_batch_jsonl

APARTES_CODEBOOK_PATH = RUN_OUTPUT_ROOT / "03_apartes" / "codebook_atos_fala.csv"
APARTES_BATCH_REQUEST_PATH = RUN_OUTPUT_ROOT / "03_apartes" / f"batch_atos_fala_{APARTES_QUALITATIVE_MODEL}.jsonl"
if GERAR_JSONL_ATOS_FALA:
    APARTES_SEGMENTATION_GATE = json.loads(APARTES_SEGMENTATION_QUALITY_PATH.read_text(encoding="utf-8"))
    assert APARTES_SEGMENTATION_GATE["classification_authorized"] is True, "A segmentação ainda não atingiu o gate."
    APARTES_CODEBOOK = pd.read_csv(APARTES_CODEBOOK_PATH).fillna("")
    APARTES_CODEBOOK_FIELDS = ["definicao_operacional", "criterio_positivo", "criterio_negativo", "caso_limitrofe"]
    assert APARTES_CODEBOOK[APARTES_CODEBOOK_FIELDS].apply(lambda column: column.str.strip().ne("").all()).all(), "Complete o codebook."
    APARTES_INTERACTIONS_FOR_BATCH = pd.read_parquet(APARTES_INTERACTIONS_PATH)
    write_qualitative_batch_jsonl(
        APARTES_INTERACTIONS_FOR_BATCH,
        APARTES_BATCH_REQUEST_PATH,
        codebook=APARTES_CODEBOOK.to_csv(index=False),
        config=ANALYSIS_CONFIG,
        model=APARTES_QUALITATIVE_MODEL,
    )
    print(APARTES_BATCH_REQUEST_PATH)
else:
    print("JSONL de atos de fala não gerado.")

## Enviar o Batch de atos de fala

O envio é uma ação separada e explícita. A chave vem do ambiente ou dos Secrets do Colab e não é gravada.

In [ ]:
import os
from openai import OpenAI
from analise.discursos_plenario.figuras import submit_responses_batch
from analise.discursos_plenario.io import write_json_atomic

APARTES_BATCH_SUBMISSION = None
if ENVIAR_BATCH_ATOS_FALA:
    assert APARTES_BATCH_REQUEST_PATH.exists(), "Gere e inspecione o JSONL primeiro."
    if not os.environ.get("OPENAI_API_KEY"):
        try:
            from google.colab import userdata
            APARTES_SECRET = userdata.get("OPENAI_API_KEY")
        except Exception:
            APARTES_SECRET = None
        if APARTES_SECRET:
            os.environ["OPENAI_API_KEY"] = APARTES_SECRET
    assert os.environ.get("OPENAI_API_KEY"), "Configure OPENAI_API_KEY no ambiente ou nos Secrets do Colab."
    APARTES_OPENAI_CLIENT = OpenAI()
    APARTES_BATCH_SUBMISSION = submit_responses_batch(
        APARTES_OPENAI_CLIENT,
        APARTES_BATCH_REQUEST_PATH,
        description=f"{RUN_ID}:atos-fala:{APARTES_QUALITATIVE_MODEL}",
    )
    APARTES_BATCH_CONTROL = {
        "batch_id": APARTES_BATCH_SUBMISSION.id,
        "model": APARTES_QUALITATIVE_MODEL,
        "request_path": str(APARTES_BATCH_REQUEST_PATH),
    }
    write_json_atomic(RUN_OUTPUT_ROOT / "03_apartes" / "batch_atos_fala.json", APARTES_BATCH_CONTROL)
    print("Batch criado:", APARTES_BATCH_SUBMISSION.id)
else:
    print("Envio desativado.")

## Baixar e analisar o Batch concluído

A saída é reconciliada por `custom_id`, gera prevalências anuais e por direção de gênero e, quando o piloto estiver adjudicado, Jaccard, F1 e kappa.

In [ ]:
import json
import os
from openai import OpenAI
from analise.discursos_plenario.apartes_qualitativos import run_qualitative_results
from analise.discursos_plenario.figuras import download_completed_batch

APARTES_BATCH_OUTPUT_PATH = RUN_OUTPUT_ROOT / "03_apartes" / f"batch_atos_fala_{APARTES_QUALITATIVE_MODEL}_output.jsonl"
APARTES_BATCH_CONTROL_PATH = RUN_OUTPUT_ROOT / "03_apartes" / "batch_atos_fala.json"
if BAIXAR_BATCH_ATOS_FALA:
    assert APARTES_BATCH_CONTROL_PATH.exists(), APARTES_BATCH_CONTROL_PATH
    if not os.environ.get("OPENAI_API_KEY"):
        try:
            from google.colab import userdata
            APARTES_DOWNLOAD_SECRET = userdata.get("OPENAI_API_KEY")
        except Exception:
            APARTES_DOWNLOAD_SECRET = None
        if APARTES_DOWNLOAD_SECRET:
            os.environ["OPENAI_API_KEY"] = APARTES_DOWNLOAD_SECRET
    assert os.environ.get("OPENAI_API_KEY"), "Configure OPENAI_API_KEY no ambiente ou nos Secrets do Colab."
    APARTES_DOWNLOAD_CLIENT = OpenAI()
    APARTES_BATCH_CONTROL_LOADED = json.loads(APARTES_BATCH_CONTROL_PATH.read_text(encoding="utf-8"))
    download_completed_batch(
        APARTES_DOWNLOAD_CLIENT,
        APARTES_BATCH_CONTROL_LOADED["batch_id"],
        APARTES_BATCH_OUTPUT_PATH,
    )
    print(APARTES_BATCH_OUTPUT_PATH)
APARTES_QUALITATIVE_RESULT = None
if PROCESSAR_BATCH_ATOS_FALA:
    assert APARTES_BATCH_OUTPUT_PATH.exists(), APARTES_BATCH_OUTPUT_PATH
    APARTES_QUALITATIVE_RESULT = run_qualitative_results(
        data_root=DATA_ROOT,
        run_id=RUN_ID,
        batch_output_path=APARTES_BATCH_OUTPUT_PATH,
        request_path=APARTES_BATCH_REQUEST_PATH,
        model=APARTES_QUALITATIVE_MODEL,
        config_path=CONFIG_PATH,
    )
    print(APARTES_QUALITATIVE_RESULT["manifest_path"])
else:
    print("Processamento da saída desativado.")